In [1]:
import sleap
sleap.disable_preallocation()
sleap.versions()
sleap.system_summary()
import cv2
import sys, os
import numpy as np
from copy import copy
sys.path.append('..')

SLEAP: 1.4.1
TensorFlow: 2.7.0
Numpy: 1.21.5
Python: 3.7.12
OS: Linux-6.8.0-51-generic-x86_64-with-debian-trixie-sid
GPUs: 2/2 available
  Device: /physical_device:GPU:0
         Available: True
       Initialized: False
     Memory growth: True
  Device: /physical_device:GPU:1
         Available: True
       Initialized: False
     Memory growth: True


In [2]:
from autoencoder.load_data_10min import load_video
from python.postprocess import *
from python.animation import *

In [3]:
original_video_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/c1_high_res_10min_track_reencoded.mp4'
# new_video_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/c1_high_res_5min_track_reencoded.mp4'
original_video = load_video(original_video_path, load_as_tensor=False)
# new_video = load_video(new_video_path, load_as_tensor=False)

print(original_video.shape)
# print(new_video.shape)

(90003, 170, 174)


In [38]:
def find_start_end_idx(original_video, new_video):
    new_vid_first_frame = new_video[0]
    new_vid_last_frame = new_video[-1]
    for i in range(original_video.shape[0]):
        if np.array_equal(original_video[i], new_vid_first_frame):
            start_idx = i
        elif np.array_equal(original_video[i], new_vid_last_frame):
            end_idx = i
            break
    print(start_idx, end_idx)
    return start_idx, end_idx

In [10]:
new_vid_first_frame = new_video[0]
new_vid_last_frame = new_video[-1]
for i in range(original_video.shape[0]):
    if np.array_equal(original_video[i], new_vid_first_frame):
        start_idx = i
    elif np.array_equal(original_video[i], new_vid_last_frame):
        end_idx = i
        break
print(start_idx, end_idx)

2850 47849


In [5]:
# start_idx, end_idx = 2977, 47935

In [11]:
old_points_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/all_tracked_points_raw_0.npy'
old_points = np.load(old_points_path)
new_points = old_points[start_idx:end_idx+1]
print(new_points.shape)

(45000, 17, 2)


In [4]:
reencoded_10min_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/predictions/flowmax-tracking/reencoded_10min_t3.slp'
reencoded_10min_labels = sleap.load_file(reencoded_10min_path)
print(reencoded_10min_labels)

Labels(labeled_frames=90003, videos=1, skeletons=1, tracks=18)


In [33]:
new_tracks = [track for track in reencoded_10min_labels.tracks if track.name != 'track_17']
new_tracks[0]

Track(spawned_on=0, name='track_0')

In [5]:
new_model_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/reencoded_5min_v2.slp'
new_model = sleap.load_file(new_model_path)

In [6]:
all_points = get_all_tracked_points(new_model, interpolate=False, min_score=0, start_idx=0)

all_tracked_points shape: (45000, 17, 2)


First non missing frame idx: 0
Missing point count: 72608
reorder_idx.shape: (17,)


In [7]:
def dataset_with_new_points(old_labels, new_points, node_name='tb1_node', handle_first_frame=True, start_idx=0):
    corrected_label = copy(old_labels)
    skl = corrected_label.skeletons[0]
    print(skl)
    instance_cnt = len(corrected_label.labeled_frames[1].instances)
    print(instance_cnt)
    
    track_name_to_idx = {}
    for trk_idx, trk in enumerate(old_labels.tracks):
        track_name = int(trk.name.split('_')[-1])
        track_name_to_idx[track_name] = trk_idx
    
    missing_pt_cnt = 0
    for lf_idx, lf in enumerate(corrected_label.labeled_frames[start_idx:], start=start_idx):
        all_instances = []
        for inst_idx in range(instance_cnt):
            x, y = new_points[lf_idx - 1, inst_idx]
            if x + y == 0:
                missing_pt_cnt += 1
                continue
            point_dict = {node_name: sleap.instance.Point(x=x, y=y)}
            curr_track = corrected_label.tracks[track_name_to_idx[inst_idx]]
            curr_track_num = int(curr_track.name.split('_')[-1])
            if curr_track_num != inst_idx:
                print(f'curr_track_num: {curr_track_num}, inst_idx: {inst_idx}')
            tb_instance = sleap.Instance(skeleton=skl, points=point_dict, frame=lf, track=curr_track)
            all_instances.append(tb_instance)
        lf.instances = all_instances
    print(f'missing_pt_cnt: {missing_pt_cnt}')
    
    if handle_first_frame:
        # special handling of the first frame
        frame_0_instances = corrected_label.labeled_frames[0].instances[:]
        frame_0_instances.pop(4)
        new_frame_0_instances = []
        lf = corrected_label.labeled_frames[0]

        for inst_0 in frame_0_instances:
            pt_0 = np.array([inst_0.points[0].x, inst_0.points[0].y])
            pts_1 = np.array([[inst.points[0].x, inst.points[0].y] 
                            for inst in corrected_label.labeled_frames[1].instances])
            distances = np.sqrt(np.sum((pts_1 - pt_0)**2, axis=1))
            nearest_idx = np.argmin(distances)
            
            x, y = pt_0
            point_dict = {node_name: sleap.instance.Point(x=x, y=y)}
            curr_track = corrected_label.tracks[track_name_to_idx[nearest_idx]]
            tb_instance = sleap.Instance(skeleton=skl, points=point_dict, frame=lf, track=curr_track)
            new_frame_0_instances.append(tb_instance)

        corrected_label.labeled_frames[0].instances = new_frame_0_instances

        # remove extract track
        for track in corrected_label.tracks:
            if track.name == 'track_17':
                corrected_label.tracks.remove(track)
                break
        
    return corrected_label

In [32]:
new_model_with_new_points = dataset_with_new_points(new_model, new_points, node_name='tb', handle_first_frame=False)
print(new_model_with_new_points)

Skeleton(description=None, nodes=[tb], edges=[], symmetries=[])
17
missing_pt_cnt: 72752
Labels(labeled_frames=45000, videos=1, skeletons=1, tracks=17)


In [37]:
save_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/reencoded_5min_v2.slp'
new_model_with_new_points.save(save_path)

In [ ]:
new_tracks = [track for track in reencoded_10min_labels.tracks if track.name != 'track_17']

for i in range(5):
        
    new_dataset_path = f'/home/mingxiao/Desktop/jellyfish/video/video_1_clips/reencoded_5min_c{i}.slp'
    new_dataset = sleap.load_file(new_dataset_path)
    # skeleton = sleap.Skeleton(name=f'TB')
    # skeleton.add_node(f'tb')
    # new_dataset.skeletons = [skeleton]
    new_dataset.tracks = new_tracks
    
    new_points = all_points[i * 9000 : (i + 1) * 9000]
    
    new_dataset_w_pts = dataset_with_new_points(new_dataset, new_points, node_name='tb1_node', handle_first_frame=False, start_idx=0)
    new_dataset_w_pts.save(new_dataset_path)


KeyError: "Unable to open object (object 'frames' doesn't exist)"